Connect to google drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Chống warnings

In [2]:
import warnings
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

LSTM model

In [3]:
import os
import gc
import time
import random
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEEDS = [42, 123, 456, 789, 2024]


In [4]:
def create_lstm_tensors(file_path, station_name):
    print(f"\nProcessing: {station_name}")

    save_dir = "/content/tensors"
    os.makedirs(save_dir, exist_ok=True)

    df = pd.read_csv(file_path)
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    df = df.dropna()

    X_df = df.copy()
    y_df = df[['wind_speed']].copy()

    n         = len(df)
    train_end = int(n * 0.7)

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    scaler_X.fit(X_df.iloc[:train_end])
    scaler_y.fit(y_df.iloc[:train_end])

    X_scaled = scaler_X.transform(X_df)
    y_scaled = scaler_y.transform(y_df)

    seq_len = 12
    X_list, y_list = [], []
    for i in range(len(X_scaled) - seq_len - 8):
        X_list.append(X_scaled[i:i+seq_len])
        y_list.append([
            y_scaled[i+seq_len+1][0],
            y_scaled[i+seq_len+3][0],
            y_scaled[i+seq_len+7][0]
        ])

    X_np = np.array(X_list)
    y_np = np.array(y_list)

    n_windows = len(X_np)
    w_train   = int(n_windows * 0.7)
    w_val     = int(n_windows * 0.85)

    tensors = (
        torch.tensor(X_np[:w_train],        dtype=torch.float32),
        torch.tensor(y_np[:w_train],        dtype=torch.float32),
        torch.tensor(X_np[w_train:w_val],   dtype=torch.float32),
        torch.tensor(y_np[w_train:w_val],   dtype=torch.float32),
        torch.tensor(X_np[w_val:],          dtype=torch.float32),
        torch.tensor(y_np[w_val:],          dtype=torch.float32),
    )
    torch.save({"tensors": tensors}, f"{save_dir}/{station_name}_tensors.pt")
    torch.save(scaler_y,             f"{save_dir}/{station_name}_scaler.pt")
    print("  Saved tensors + scaler")


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, output_size=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.fc   = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, output_size)
        )

    def forward(self, x):
        h0 = torch.zeros(self.lstm.num_layers, x.size(0),
                         self.lstm.hidden_size).to(x.device)
        c0 = torch.zeros(self.lstm.num_layers, x.size(0),
                         self.lstm.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def measure_latency_dl(model, n_warmup=100, n_measure=500, seq_len=12, n_features=22):
    model_cpu = model.cpu().eval()
    x = torch.randn(1, seq_len, n_features)
    with torch.no_grad():
        for _ in range(n_warmup):
            model_cpu(x)
        lats = []
        for _ in range(n_measure):
            t0 = time.perf_counter()
            model_cpu(x)
            lats.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(lats)), float(np.std(lats))


def train_lstm(station_name, seed):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data   = torch.load(f"/content/tensors/{station_name}_tensors.pt", weights_only=False)
    scaler = torch.load(f"/content/tensors/{station_name}_scaler.pt",  weights_only=False)
    X_train, y_train, X_val, y_val, X_test, y_test = data["tensors"]

    X_train, y_train = X_train.to(device), y_train.to(device)
    X_val,   y_val   = X_val.to(device),   y_val.to(device)
    X_test,  y_test  = X_test.to(device),  y_test.to(device)

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=64, shuffle=False)

    model     = LSTMModel(X_train.shape[2]).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5)

    best_loss = float('inf')
    patience  = 10
    counter   = 0
    save_path = f"/content/{station_name}_best_lstm.pt"

    t_start = time.time()
    for epoch in range(50):
        model.train()
        epoch_loss = 0
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        train_loss = epoch_loss / len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb_v, yb_v in val_loader:
                val_loss += criterion(model(Xb_v), yb_v).item()
        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            counter   = 0
            torch.save(model.state_dict(), save_path)
        else:
            counter += 1
        if counter >= patience:
            break
    train_time_s = time.time() - t_start

    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    y_pred_list = []
    with torch.no_grad():
        for Xb_t, _ in DataLoader(TensorDataset(X_test, y_test),
                                   batch_size=256, shuffle=False):
            y_pred_list.append(model(Xb_t).cpu().numpy())

    y_pred_scaled = np.vstack(y_pred_list)
    y_true_scaled = y_test.cpu().numpy()
    y_pred = scaler.inverse_transform(y_pred_scaled)
    y_true = scaler.inverse_transform(y_true_scaled)

    horizons = ["30min", "60min", "120min"]
    metrics  = {}
    for i, h in enumerate(horizons):
        metrics[h] = {
            "rmse": float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))),
            "mae":  float(mean_absolute_error(y_true[:, i], y_pred[:, i])),
            "r2":   float(r2_score(y_true[:, i], y_pred[:, i])),
        }

    n_feat   = X_train.shape[2]
    n_p      = count_params(model)
    lat_m, lat_s = measure_latency_dl(model, seq_len=12, n_features=n_feat)

    metrics["__meta__"] = {
        "train_time_s":    train_time_s,
        "latency_mean_ms": lat_m,
        "latency_std_ms":  lat_s,
        "n_params":        n_p,
    }
    return metrics

In [5]:
stations  = [
    "Site A - Inland",
    "Site B - Coastal",
    "Site C - Complex Terrain",
    "Site D - Offshore",
]
base_path = '/content/drive/MyDrive/wind_forecast/'
horizons  = ["30min", "60min", "120min"]

for station in stations:
    create_lstm_tensors(f"{base_path}{station}.csv", station)

from collections import defaultdict
lstm_results = defaultdict(lambda: defaultdict(list))
lstm_meta    = defaultdict(list)   # {station: list of __meta__ dicts}

print("LSTM — MULTI-SEED EVALUATION  (5 seeds × 4 sites)")

start_all_lstm = time.time()

for station in stations:
    print(f"\nStation: {station}")
    for seed in SEEDS:
        print(f"Seed {seed}...", end=" ", flush=True)
        try:
            metrics = train_lstm(station, seed)
            meta    = metrics.pop("__meta__")
            for h in horizons:
                lstm_results[station][h].append(metrics[h])
            lstm_meta[station].append(meta)
            r2s = [f'{metrics[h]["r2"]:.4f}' for h in horizons]
            print(f"R²={r2s}  lat={meta['latency_mean_ms']:.2f}ms  t={meta['train_time_s']:.0f}s")
        except Exception as e:
            print(f"ERROR: {e}")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

total_lstm_min = (time.time() - start_all_lstm) / 60

print("LSTM — SUMMARY  (mean ± std, 5 seeds)")
for station in stations:
    print(f"\n  {station}")
    for h in horizons:
        rmses = [m["rmse"] for m in lstm_results[station][h]]
        maes  = [m["mae"]  for m in lstm_results[station][h]]
        r2s   = [m["r2"]   for m in lstm_results[station][h]]
        print(f"    {h:7s}  RMSE={np.mean(rmses):.4f}±{np.std(rmses):.4f}"
              f"  MAE={np.mean(maes):.4f}±{np.std(maes):.4f}"
              f"  R²={np.mean(r2s):.4f}±{np.std(r2s):.4f}")
    lats = [m["latency_mean_ms"] for m in lstm_meta[station]]
    ts   = [m["train_time_s"]    for m in lstm_meta[station]]
    np_  =  lstm_meta[station][0]["n_params"]
    print(f"    Params   : {np_:,}  ({np_/1e3:.1f} K)")
    print(f"    Latency  : {np.mean(lats):.3f}±{np.std(lats):.3f} ms/sample (CPU, batch=1)")
    print(f"    TrainTime: {np.mean(ts):.1f}±{np.std(ts):.1f} s/seed")

print(f"\nTotal wall-clock: {total_lstm_min:.2f} min")


Processing: Site A - Inland
  Saved tensors + scaler

Processing: Site B - Coastal
  Saved tensors + scaler

Processing: Site C - Complex Terrain
  Saved tensors + scaler

Processing: Site D - Offshore
  Saved tensors + scaler
LSTM — MULTI-SEED EVALUATION  (5 seeds × 4 sites)

Station: Site A - Inland
Seed 42... R²=['0.9897', '0.9610', '0.8702']  lat=0.79ms  t=174s
Seed 123... R²=['0.9889', '0.9590', '0.8679']  lat=0.50ms  t=150s
Seed 456... R²=['0.9905', '0.9627', '0.8768']  lat=0.53ms  t=163s
Seed 789... R²=['0.9881', '0.9609', '0.8739']  lat=0.56ms  t=122s
Seed 2024... R²=['0.9887', '0.9602', '0.8718']  lat=0.52ms  t=209s

Station: Site B - Coastal
Seed 42... R²=['0.9974', '0.9887', '0.9634']  lat=0.78ms  t=200s
Seed 123... R²=['0.9973', '0.9888', '0.9639']  lat=0.55ms  t=173s
Seed 456... R²=['0.9974', '0.9888', '0.9635']  lat=0.48ms  t=209s
Seed 789... R²=['0.9975', '0.9888', '0.9640']  lat=0.55ms  t=186s
Seed 2024... R²=['0.9975', '0.9886', '0.9625']  lat=0.52ms  t=238s

Station:

##Table 4 / 5 / 6 — LSTM Performance & Benchmark Summary

In [6]:
import numpy as np
N_SEEDS_L = len(SEEDS)

print("TABLE 4 — LSTM  |  mean ± std, 5 seeds, averaged across 4 sites")
print(f"  {'Horizon':<10}  {'RMSE':>20}  {'MAE':>20}  {'R²':>20}")

for h in horizons:
    seed_rmse = [np.mean([lstm_results[st][h][i]["rmse"] for st in stations])
                 for i in range(N_SEEDS_L)]
    seed_mae  = [np.mean([lstm_results[st][h][i]["mae"]  for st in stations])
                 for i in range(N_SEEDS_L)]
    seed_r2   = [np.mean([lstm_results[st][h][i]["r2"]   for st in stations])
                 for i in range(N_SEEDS_L)]
    print(f"  {h:<10}  {np.mean(seed_rmse):.4f} ± {np.std(seed_rmse):.4f}"
          f"  {np.mean(seed_mae):.4f} ± {np.std(seed_mae):.4f}"
          f"  {np.mean(seed_r2):.4f} ± {np.std(seed_r2):.4f}")

print("TABLE 5 — LSTM  |  site-wise 30-min performance  (mean ± std, 5 seeds)")
site_labels = {
    "Site A - Inland":            "A (Inland)",
    "Site B - Coastal":           "B (Coastal)",
    "Site C - Complex Terrain":   "C (Complex Terrain)",
    "Site D - Offshore":          "D (Offshore)",
}
print(f"  {'Site':<26}  {'RMSE':>22}  {'MAE':>22}  {'R²':>22}")
for st in stations:
    rmses = [lstm_results[st]["30min"][i]["rmse"] for i in range(N_SEEDS_L)]
    maes  = [lstm_results[st]["30min"][i]["mae"]  for i in range(N_SEEDS_L)]
    r2s   = [lstm_results[st]["30min"][i]["r2"]   for i in range(N_SEEDS_L)]
    print(f"  {site_labels[st]:<26}"
          f"  {np.mean(rmses):.4f}±{np.std(rmses):.4f}"
          f"  {np.mean(maes):.4f}±{np.std(maes):.4f}"
          f"  {np.mean(r2s):.4f}±{np.std(r2s):.4f}")

all_lats   = [m["latency_mean_ms"] for st in stations for m in lstm_meta[st]]
all_times  = [m["train_time_s"]    for st in stations for m in lstm_meta[st]]
n_params_l = lstm_meta[stations[0]][0]["n_params"]
footprint_mb = n_params_l * 4 / 1e6   # FP32

# Global RMSE
global_rmse_lstm = float(np.mean([
    lstm_results[st][h][i]["rmse"]
    for st in stations for h in horizons for i in range(N_SEEDS_L)
]))

print("TABLE 6 — LSTM  |  Model complexity & operational deployment")
print(f"  Parameters        : {n_params_l:,}  ({n_params_l/1e3:.1f} K)")
print(f"  Model footprint   : ~{footprint_mb:.2f} MB  (FP32 weights)")
print(f"  Latency (CPU)     : {np.mean(all_lats):.3f} ± {np.std(all_lats):.3f} ms/sample"
      f"  (100 warm-up + 500 measure, batch=1)")
print(f"  Avg train time    : {np.mean(all_times):.1f} ± {np.std(all_times):.1f} s/seed")
print(f"  Total train time  : {total_lstm_min:.2f} min  (all sites × all seeds)")
print(f"  Global RMSE       : {global_rmse_lstm:.4f}  (avg all sites × horizons × seeds)")

TABLE 4 — LSTM  |  mean ± std, 5 seeds, averaged across 4 sites
  Horizon                     RMSE                   MAE                    R²
  30min       0.1868 ± 0.0021  0.1203 ± 0.0039  0.9948 ± 0.0002
  60min       0.3714 ± 0.0018  0.2497 ± 0.0014  0.9798 ± 0.0003
  120min      0.6732 ± 0.0020  0.4863 ± 0.0011  0.9349 ± 0.0008
TABLE 5 — LSTM  |  site-wise 30-min performance  (mean ± std, 5 seeds)
  Site                                          RMSE                     MAE                      R²
  A (Inland)                  0.2083±0.0081  0.1273±0.0152  0.9892±0.0008
  B (Coastal)                 0.1504±0.0019  0.1032±0.0021  0.9974±0.0001
  C (Complex Terrain)         0.1961±0.0050  0.1302±0.0067  0.9948±0.0003
  D (Offshore)                0.1923±0.0102  0.1205±0.0055  0.9978±0.0002
TABLE 6 — LSTM  |  Model complexity & operational deployment
  Parameters        : 58,243  (58.2 K)
  Model footprint   : ~0.23 MB  (FP32 weights)
  Latency (CPU)     : 0.570 ± 0.099 ms/sample  (10

In [7]:
import os
preds_lstm = {}   # {station_name: {'y_true': ndarray, 'y_pred': ndarray}}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for station in stations:
    set_seed(42)
    tensors_path = f"/content/tensors/{station}_tensors.pt"
    scaler_path  = f"/content/tensors/{station}_scaler.pt"
    model_path   = f"/content/{station}_best_lstm.pt"

    if not all(os.path.exists(p) for p in [tensors_path, scaler_path, model_path]):
        print(f"  [SKIP] missing saved files for {station}")
        continue

    data    = torch.load(tensors_path, weights_only=False)
    scaler  = torch.load(scaler_path,  weights_only=False)
    _, _, _, _, X_test, y_test = data["tensors"]

    model = LSTMModel(X_test.shape[2]).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    with torch.no_grad():
        y_pred_sc = model(X_test.to(device)).cpu().numpy()

    y_pred = scaler.inverse_transform(y_pred_sc)
    y_true = scaler.inverse_transform(y_test.numpy())
    preds_lstm[station] = {'y_true': y_true, 'y_pred': y_pred}
    print(f"  {station}: test samples = {len(y_true)}")

PRED_DIR = '/content/drive/MyDrive/wind_forecast/'
os.makedirs(PRED_DIR, exist_ok=True)

np.save('/content/preds_lstm.npy',       preds_lstm)
np.save(PRED_DIR + 'preds_lstm.npy',     preds_lstm)

# Per-seed RMSE → Figure 5 boxplot trong Mamba notebook
if 'lstm_results' in dir():
    rmse_lstm_per_seed = {
        st: {h: [lstm_results[st][h][i]['rmse'] for i in range(len(SEEDS))]
             for h in horizons}
        for st in stations
    }
    np.save('/content/rmse_lstm_per_seed.npy',   rmse_lstm_per_seed)
    np.save(PRED_DIR + 'rmse_lstm_per_seed.npy', rmse_lstm_per_seed)
    print(f'Saved → {PRED_DIR}rmse_lstm_per_seed.npy')
else:
    print('[SKIP] rmse_lstm_per_seed: run the eval loop cell first')

print(f'Saved → /content/preds_lstm.npy')
print(f'Saved → {PRED_DIR}preds_lstm.npy')
print('Keys per station: y_true[:,0]=30min  y_true[:,1]=60min  y_true[:,2]=120min')

  Site A - Inland: test samples = 15776
  Site B - Coastal: test samples = 15776
  Site C - Complex Terrain: test samples = 15761
  Site D - Offshore: test samples = 15776
Saved → /content/drive/MyDrive/wind_forecast/rmse_lstm_per_seed.npy
Saved → /content/preds_lstm.npy
Saved → /content/drive/MyDrive/wind_forecast/preds_lstm.npy
Keys per station: y_true[:,0]=30min  y_true[:,1]=60min  y_true[:,2]=120min


Transformer model

In [8]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
torch.manual_seed(42)

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)


class WindTransformer(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4,
                 num_layers=2, seq_len=12):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos        = PositionalEncoding(d_model, seq_len)
        encoder_layer   = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(encoder_layer,
                                             num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.fc   = nn.Linear(d_model, 3)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos(x)
        x = self.encoder(x)
        x = self.norm(x)
        return self.fc(x[:, -1, :])


def train_transformer(file_path, station_name, seed):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path)
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    df = df.dropna()

    n         = len(df)
    train_end = int(n * 0.7)
    X_df      = df.copy()
    Y_df      = df[['wind_speed']]

    scaler_X = MinMaxScaler(); scaler_Y = MinMaxScaler()
    scaler_X.fit(X_df.iloc[:train_end])
    scaler_Y.fit(Y_df.iloc[:train_end])

    X_scaled = scaler_X.transform(X_df)
    Y_scaled = scaler_Y.transform(Y_df)

    seq_len = 12
    X_list, Y_list = [], []
    for i in range(len(X_scaled) - seq_len - 10):
        X_list.append(X_scaled[i:i+seq_len])
        Y_list.append([
            Y_scaled[i+seq_len+1][0],
            Y_scaled[i+seq_len+3][0],
            Y_scaled[i+seq_len+7][0],
        ])

    X_np = np.array(X_list, dtype=np.float32)
    Y_np = np.array(Y_list, dtype=np.float32)

    n_w     = len(X_np)
    w_train = int(n_w * 0.70)
    w_val   = int(n_w * 0.85)

    X_train = torch.tensor(X_np[:w_train]).to(device)
    Y_train = torch.tensor(Y_np[:w_train]).to(device)
    X_val   = torch.tensor(X_np[w_train:w_val]).to(device)
    Y_val   = torch.tensor(Y_np[w_train:w_val]).to(device)
    X_test  = torch.tensor(X_np[w_val:]).to(device)
    Y_test  = torch.tensor(Y_np[w_val:]).to(device)

    train_loader = DataLoader(TensorDataset(X_train, Y_train),
                              batch_size=64, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val,   Y_val),
                              batch_size=64, shuffle=False)

    model     = WindTransformer(X_train.shape[2], seq_len=seq_len).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5)

    best_loss = float('inf')
    patience  = 10
    counter   = 0
    save_path = f"/content/{station_name}_best_transformer.pt"

    t_start = time.time()
    for epoch in range(50):
        model.train()
        epoch_loss = 0
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        train_loss = epoch_loss / len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb_v, yb_v in val_loader:
                val_loss += criterion(model(Xb_v), yb_v).item()
        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss; counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            counter += 1
        if counter >= patience:
            break
    train_time_s = time.time() - t_start

    # ── Evaluate ──
    model.load_state_dict(torch.load(save_path))
    model.eval()
    with torch.no_grad():
        y_pred_list = []
        for Xb_t, _ in DataLoader(TensorDataset(X_test, Y_test),
                                   batch_size=256, shuffle=False):
            y_pred_list.append(model(Xb_t).cpu().numpy())

    y_pred_scaled = np.vstack(y_pred_list)
    y_true_scaled = Y_test.cpu().numpy()
    y_pred = scaler_Y.inverse_transform(y_pred_scaled)
    y_true = scaler_Y.inverse_transform(y_true_scaled)

    horizons = ["30min", "60min", "120min"]
    metrics  = {}
    for i, h in enumerate(horizons):
        metrics[h] = {
            "rmse": float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))),
            "mae":  float(mean_absolute_error(y_true[:, i], y_pred[:, i])),
            "r2":   float(r2_score(y_true[:, i], y_pred[:, i])),
        }
    metrics["__meta__"] = {
        "train_time_s":    train_time_s,
        "n_params":        count_params(model),
    }
    return metrics


# ── Helper utilities for Table 6 benchmark ──────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def measure_latency_dl(model, n_warmup=100, n_measure=500, seq_len=12, n_features=22):
    model_cpu = model.cpu().eval()
    x = torch.randn(1, seq_len, n_features)
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model_cpu(x)
        lats = []
        for _ in range(n_measure):
            t0 = time.perf_counter()
            _ = model_cpu(x)
            lats.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(lats)), float(np.std(lats))


In [10]:
stations  = [
    "Site A - Inland",
    "Site B - Coastal",
    "Site C - Complex Terrain",
    "Site D - Offshore",
]
base_path = '/content/drive/MyDrive/wind_forecast/'
horizons  = ["30min", "60min", "120min"]

from collections import defaultdict
trans_results = defaultdict(lambda: defaultdict(list))
trans_meta    = defaultdict(list)

print("Transformer — MULTI-SEED EVALUATION  (5 seeds × 4 sites)")

start_all_trans = time.time()

for station in stations:
    path = f"{base_path}{station}.csv"
    print(f"\nStation: {station}")
    for seed in SEEDS:
        print(f"Seed {seed}...", end=" ", flush=True)
        try:
            metrics = train_transformer(path, station, seed)
            meta    = metrics.pop("__meta__")
            for h in horizons:
                trans_results[station][h].append(metrics[h])
            trans_meta[station].append(meta)
            r2s = [f'{metrics[h]["r2"]:.4f}' for h in horizons]
            print(f"R²={r2s}  t={meta['train_time_s']:.0f}s")
        except Exception as e:
            print(f"ERROR: {e}")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

total_trans_min = (time.time() - start_all_trans) / 60

print("Transformer — SUMMARY  (mean ± std, 5 seeds)")
for station in stations:
    print(f"\n  {station}")
    for h in horizons:
        rmses = [m["rmse"] for m in trans_results[station][h]]
        maes  = [m["mae"]  for m in trans_results[station][h]]
        r2s   = [m["r2"]   for m in trans_results[station][h]]
        print(f"    {h:7s}  RMSE={np.mean(rmses):.4f}±{np.std(rmses):.4f}"
              f"  MAE={np.mean(maes):.4f}±{np.std(maes):.4f}"
              f"  R²={np.mean(r2s):.4f}±{np.std(r2s):.4f}")
    ts  = [m["train_time_s"] for m in trans_meta[station]]
    np_ =  trans_meta[station][0]["n_params"]
    print(f"    Params   : {np_:,}  ({np_/1e3:.1f} K)")
    print(f"    TrainTime: {np.mean(ts):.1f} ± {np.std(ts):.1f} s/seed")

print(f"\nTotal wall-clock: {total_trans_min:.2f} min")

Transformer — MULTI-SEED EVALUATION  (5 seeds × 4 sites)

Station: Site A - Inland
Seed 42... R²=['0.9870', '0.9615', '0.8800']  t=224s
Seed 123... R²=['0.9915', '0.9650', '0.8808']  t=221s
Seed 456... R²=['0.9915', '0.9641', '0.8775']  t=205s
Seed 789... R²=['0.9812', '0.9529', '0.8758']  t=223s
Seed 2024... R²=['0.9902', '0.9637', '0.8836']  t=157s

Station: Site B - Coastal
Seed 42... R²=['0.9973', '0.9883', '0.9608']  t=294s
Seed 123... R²=['0.9973', '0.9881', '0.9616']  t=247s
Seed 456... R²=['0.9972', '0.9884', '0.9631']  t=216s
Seed 789... R²=['0.9972', '0.9887', '0.9629']  t=255s
Seed 2024... R²=['0.9976', '0.9889', '0.9636']  t=222s

Station: Site C - Complex Terrain
Seed 42... R²=['0.9928', '0.9733', '0.9285']  t=213s
Seed 123... R²=['0.9950', '0.9785', '0.9355']  t=230s
Seed 456... R²=['0.9952', '0.9779', '0.9343']  t=220s
Seed 789... R²=['0.9946', '0.9771', '0.9317']  t=245s
Seed 2024... R²=['0.9946', '0.9762', '0.9333']  t=223s

Station: Site D - Offshore
Seed 42... R²=['0

In [11]:
import os
preds_transformer = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for station in stations:
    set_seed(42)
    path = f"{base_path}{station}.csv"
    model_path = f"/content/{station}_best_transformer.pt"

    if not os.path.exists(model_path):
        print(f"  [SKIP] no saved model for {station}")
        continue

    # Rebuild test set (same as train_transformer logic)
    df = pd.read_csv(path)
    df['time'] = pd.to_datetime(df['time']); df.set_index('time', inplace=True); df = df.dropna()
    n = len(df); train_end = int(n * 0.7)
    X_df = df.copy(); Y_df = df[['wind_speed']]
    scaler_X = MinMaxScaler(); scaler_Y = MinMaxScaler()
    scaler_X.fit(X_df.iloc[:train_end]); scaler_Y.fit(Y_df.iloc[:train_end])
    X_sc = scaler_X.transform(X_df); Y_sc = scaler_Y.transform(Y_df)
    seq_len = 12
    X_list, Y_list = [], []
    for i in range(len(X_sc) - seq_len - 10):
        X_list.append(X_sc[i:i+seq_len])
        Y_list.append([Y_sc[i+seq_len+1][0], Y_sc[i+seq_len+3][0], Y_sc[i+seq_len+7][0]])
    X_np = np.array(X_list, dtype=np.float32); Y_np = np.array(Y_list, dtype=np.float32)
    n_w = len(X_np); w_val = int(n_w * 0.85)
    X_test_np = X_np[w_val:]; Y_test_np  = Y_np[w_val:]

    model = WindTransformer(X_test_np.shape[2], seq_len=seq_len).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    X_test_t = torch.tensor(X_test_np).to(device)
    with torch.no_grad():
        y_pred_sc = model(X_test_t).cpu().numpy()

    y_pred = scaler_Y.inverse_transform(y_pred_sc)
    y_true = scaler_Y.inverse_transform(Y_test_np)
    preds_transformer[station] = {'y_true': y_true, 'y_pred': y_pred}
    print(f"  {station}: test samples = {len(y_true)}")

PRED_DIR = '/content/drive/MyDrive/wind_forecast/'
os.makedirs(PRED_DIR, exist_ok=True)

np.save('/content/preds_transformer.npy',          preds_transformer)
np.save(PRED_DIR + 'preds_transformer.npy',        preds_transformer)

# Per-seed RMSE dict → dùng cho Figure 5 boxplot trong Mamba notebook
if 'trans_results' in dir():
    rmse_trans_per_seed = {
        st: {h: [trans_results[st][h][i]['rmse'] for i in range(len(SEEDS))]
             for h in horizons}
        for st in stations
    }
    np.save('/content/rmse_trans_per_seed.npy',   rmse_trans_per_seed)
    np.save(PRED_DIR + 'rmse_trans_per_seed.npy', rmse_trans_per_seed)
    print(f'Saved → {PRED_DIR}rmse_trans_per_seed.npy')
else:
    print('[SKIP] rmse_trans_per_seed: run the eval loop cell first')

  Site A - Inland: test samples = 15776
  Site B - Coastal: test samples = 15776
  Site C - Complex Terrain: test samples = 15761
  Site D - Offshore: test samples = 15776
Saved → /content/drive/MyDrive/wind_forecast/rmse_trans_per_seed.npy


In [12]:
site_code_map = {
    'Site A - Inland'         : 'A',
    'Site B - Coastal'        : 'B',
    'Site C - Complex Terrain': 'C',
    'Site D - Offshore'       : 'D',
}

# ── LSTM ──
rmse_lstm_per_seed = {}
for st in stations:
    c = site_code_map[st]
    rmse_lstm_per_seed[c] = {}
    for h in horizons:
        rmse_lstm_per_seed[c][h] = [m['rmse'] for m in lstm_results[st][h]]

np.save('/content/drive/MyDrive/wind_forecast/rmse_lstm_per_seed.npy', rmse_lstm_per_seed)
print("Saved : rmse_lstm_per_seed.npy")

# ── Transformer ──
rmse_trans_per_seed = {}
for st in stations:
    c = site_code_map[st]
    rmse_trans_per_seed[c] = {}
    for h in horizons:
        rmse_trans_per_seed[c][h] = [m['rmse'] for m in trans_results[st][h]]

np.save('/content/drive/MyDrive/wind_forecast/rmse_trans_per_seed.npy', rmse_trans_per_seed)
print("Saved: rmse_trans_per_seed.npy")

# Quick sanity check
for model_name, d in [('LSTM', rmse_lstm_per_seed), ('Transformer', rmse_trans_per_seed)]:
    for c in ['A','B','C','D']:
        v = d[c]['30min']
        print(f"  {model_name} Site {c} 30min: mean={np.mean(v):.4f}  n={len(v)}")

Saved : rmse_lstm_per_seed.npy
Saved: rmse_trans_per_seed.npy
  LSTM Site A 30min: mean=0.2083  n=5
  LSTM Site B 30min: mean=0.1504  n=5
  LSTM Site C 30min: mean=0.1961  n=5
  LSTM Site D 30min: mean=0.1923  n=5
  Transformer Site A 30min: mean=0.2143  n=5
  Transformer Site B 30min: mean=0.1531  n=5
  Transformer Site C 30min: mean=0.2015  n=5
  Transformer Site D 30min: mean=0.1742  n=5


In [13]:
import numpy as np
N_SEEDS_T = len(SEEDS)

print("TABLE 4 — Transformer  |  mean ± std, 5 seeds, averaged across 4 sites")
print(f"  {'Horizon':<10}  {'RMSE':>20}  {'MAE':>20}  {'R²':>20}")

for h in horizons:
    seed_rmse = [np.mean([trans_results[st][h][i]["rmse"] for st in stations])
                 for i in range(N_SEEDS_T)]
    seed_mae  = [np.mean([trans_results[st][h][i]["mae"]  for st in stations])
                 for i in range(N_SEEDS_T)]
    seed_r2   = [np.mean([trans_results[st][h][i]["r2"]   for st in stations])
                 for i in range(N_SEEDS_T)]
    print(f"  {h:<10}  {np.mean(seed_rmse):.4f} ± {np.std(seed_rmse):.4f}"
          f"  {np.mean(seed_mae):.4f} ± {np.std(seed_mae):.4f}"
          f"  {np.mean(seed_r2):.4f} ± {np.std(seed_r2):.4f}")

print("TABLE 5 — Transformer  |  site-wise 30-min  (mean ± std, 5 seeds)")
site_labels = {
    "Site A - Inland":            "A (Inland)",
    "Site B - Coastal":           "B (Coastal)",
    "Site C - Complex Terrain":   "C (Complex Terrain)",
    "Site D - Offshore":          "D (Offshore)",
}
print(f"  {'Site':<26}  {'RMSE':>22}  {'MAE':>22}  {'R²':>22}")
for st in stations:
    rmses = [trans_results[st]["30min"][i]["rmse"] for i in range(N_SEEDS_T)]
    maes  = [trans_results[st]["30min"][i]["mae"]  for i in range(N_SEEDS_T)]
    r2s   = [trans_results[st]["30min"][i]["r2"]   for i in range(N_SEEDS_T)]
    print(f"  {site_labels[st]:<26}"
          f"  {np.mean(rmses):.4f}±{np.std(rmses):.4f}"
          f"  {np.mean(maes):.4f}±{np.std(maes):.4f}"
          f"  {np.mean(r2s):.4f}±{np.std(r2s):.4f}")

n_features_t    = 22
sample_trans    = WindTransformer(input_size=n_features_t, seq_len=12)
n_params_t      = count_params(sample_trans)
lat_m_t, lat_s_t = measure_latency_dl(sample_trans, seq_len=12, n_features=n_features_t)
footprint_mb_t  = n_params_t * 4 / 1e6

all_times_t = [m["train_time_s"] for st in stations for m in trans_meta[st]]

global_rmse_trans = float(np.mean([
    trans_results[st][h][i]["rmse"]
    for st in stations for h in horizons for i in range(N_SEEDS_T)
]))

print("TABLE 6 — Transformer  |  Model complexity & operational deployment")
print(f"  Parameters        : {n_params_t:,}  ({n_params_t/1e3:.1f} K)")
print(f"  Model footprint   : ~{footprint_mb_t:.2f} MB  (FP32 weights)")
print(f"  Latency (CPU)     : {lat_m_t:.3f} ± {lat_s_t:.3f} ms/sample"
      f"  (100 warm-up + 500 measure, batch=1)")
print(f"  Avg train time    : {np.mean(all_times_t):.1f} ± {np.std(all_times_t):.1f} s/seed")
print(f"  Total train time  : {total_trans_min:.2f} min  (all sites × all seeds)")
print(f"  Global RMSE       : {global_rmse_trans:.4f}  (avg all sites × horizons × seeds)")


TABLE 4 — Transformer  |  mean ± std, 5 seeds, averaged across 4 sites
  Horizon                     RMSE                   MAE                    R²
  ----------------------------------------------------------------------
  30min       0.1858 ± 0.0102  0.1176 ± 0.0037  0.9946 ± 0.0011
  60min       0.3732 ± 0.0064  0.2516 ± 0.0028  0.9796 ± 0.0012
  120min      0.6766 ± 0.0052  0.4910 ± 0.0021  0.9355 ± 0.0010
TABLE 5 — Transformer  |  site-wise 30-min  (mean ± std, 5 seeds)
  Site                                          RMSE                     MAE                      R²
  A (Inland)                  0.2143±0.0343  0.1308±0.0099  0.9883±0.0039
  B (Coastal)                 0.1531±0.0036  0.1037±0.0046  0.9973±0.0001
  C (Complex Terrain)         0.2015±0.0150  0.1239±0.0041  0.9944±0.0009
  D (Offshore)                0.1742±0.0069  0.1119±0.0092  0.9982±0.0001
TABLE 6 — Transformer  |  Model complexity & operational deployment
  Parameters        : 564,099  (564.1 K)
  Model foot